# SAVi (Slot Attention for Video) — MOVi-A での再現実装（PyTorch）
このノートブックは、SlotAttention ベースライン（静止画AE）を拡張して、**SAVi (Slot Attention for Video)** の「予測（predict）→補正（correct）」型の**再帰スロット更新**を実装。

- データ：**MOVi-A (TFDS, Kubric 公開GCS)**
- 学習：まずは **フレーム再構成（reconstruction）** を最小目的として学習（光流/深度などの自己教師は後で拡張可能）
- 出力：各フレームの recon、slot masks、slot latent（時間方向に一貫しやすい）

## 1. インストール

In [ ]:
!pip -q install -U tensorflow-datasets tensorflow-io-gcs-filesystem einops matplotlib tqdm
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
print("done")

## 2. import・乱数固定・GPU確認

In [ ]:
import os, time, json, math, random
import numpy as np
import tensorflow_datasets as tfds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

from einops import rearrange
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 3. MOVi-A をロード（GCSの生成済みTFDSから）

In [ ]:
import tensorflow_datasets as tfds

DATASET_NAME = "kubric:movi_a"

# 公開GCSの生成済みTFDSを複数バージョン候補で探索
GCS_BASE = "gs://kubric-public/tfds/movi_a"
GCS_VERSION_CANDIDATES = ["1.0.0", "1.0.1", "1.0.2", "1.1.0", "1.2.0"]

def try_builder_from_gcs():
    for ver in GCS_VERSION_CANDIDATES:
        gcs_dir = f"{GCS_BASE}/{ver}"
        try:
            b = tfds.builder_from_directory(gcs_dir)
            print(f"[OK] builder_from_directory: {gcs_dir}")
            return b
        except Exception as e:
            print(f"[NG] {gcs_dir} -> {type(e).__name__}: {e}")
    return None

builder = try_builder_from_gcs()

if builder is None:
    print("\nGCSから読めないため、ローカルで download_and_prepare() します（初回は重いです）")
    builder = tfds.builder(DATASET_NAME)
    builder.download_and_prepare()

info = builder.info
print("\n=== Dataset Info ===")
print(info)

## 4. train/val を作成 & キー確認（video_key / mask_key 自動推定）

In [ ]:
# --- train/val の作成 ---
train_tfds = builder.as_dataset(split="train", shuffle_files=True)
val_split = "validation" if "validation" in builder.info.splits else ("val" if "val" in builder.info.splits else None)
val_tfds = builder.as_dataset(split=val_split, shuffle_files=False) if val_split else None

# --- 1サンプルでキーとshape/dtypeを確認 ---
ex = next(iter(tfds.as_numpy(train_tfds.take(1))))
print("keys:", list(ex.keys()))
for k, v in ex.items():
    if hasattr(v, "shape"):
        print(f"{k:25s}", v.shape, v.dtype)
    else:
        print(f"{k:25s}", type(v))

# --- video_key 推定（堅牢化：名前優先 + dtype/shape優先）---
def score_video_key(k, v):
    # v: numpy array
    if not (hasattr(v, "shape") and len(v.shape) == 4 and v.shape[-1] == 3):
        return -1e9
    score = 0.0
    name = k.lower()

    # 名前の強い優先度（RGB/video/image/frames）
    if "video" in name or "rgb" in name or "image" in name or "frames" in name:
        score += 100.0
    # 逆に避けたい候補（法線/座標/メタ等）
    if "normal" in name or "xyz" in name or "coord" in name or "world" in name:
        score -= 50.0
    if "depth" in name or "seg" in name or "mask" in name:
        score -= 100.0

    # dtype が uint8 ならほぼRGB
    if v.dtype == np.uint8:
        score += 50.0

    # 値レンジがそれっぽいなら加点（サンプルだけでOK）
    try:
        vv = v[0]  # 1フレーム
        vmax = float(vv.max())
        vmin = float(vv.min())
        # RGBなら大抵 0..255 or 0..1
        if 0.0 <= vmin and vmax <= 1.0:
            score += 10.0
        if 0.0 <= vmin and vmax <= 255.0:
            score += 10.0
    except Exception:
        pass
    return score

video_candidates = []
for k, v in ex.items():
    if hasattr(v, "shape") and len(v.shape) == 4 and v.shape[-1] == 3:
        video_candidates.append((score_video_key(k, v), k, v.shape, v.dtype))

video_candidates.sort(reverse=True, key=lambda x: x[0])
print("\n[video candidates ranking]")
for s, k, sh, dt in video_candidates[:10]:
    print(f"score={s:6.1f}  {k:25s}  shape={sh}  dtype={dt}")

video_key = video_candidates[0][1] if len(video_candidates) else None

# --- mask_key 推定（インスタンスIDマップ優先）---
def score_mask_key(k, v, video_key):
    if not hasattr(v, "shape"):
        return -1e9
    name = k.lower()
    score = 0.0
    if k == video_key:
        return -1e9
    # shape: (T,H,W) or (T,H,W,1)
    if len(v.shape) == 4 and v.shape[-1] == 1:
        score += 20.0
    elif len(v.shape) == 3:
        score += 10.0
    else:
        return -1e9

    # 名前優先
    if "seg" in name or "mask" in name or "instance" in name or "id" in name:
        score += 100.0
    if "depth" in name or "flow" in name or "normal" in name:
        score -= 50.0

    # dtype が整数なら加点
    if v.dtype in (np.int32, np.int64, np.uint8, np.uint16):
        score += 20.0
    return score

mask_candidates = []
for k, v in ex.items():
    sc = score_mask_key(k, v, video_key)
    if sc > -1e8:
        mask_candidates.append((sc, k, v.shape, getattr(v, "dtype", None)))

mask_candidates.sort(reverse=True, key=lambda x: x[0])
print("\n[mask candidates ranking]")
for s, k, sh, dt in mask_candidates[:10]:
    print(f"score={s:6.1f}  {k:25s}  shape={sh}  dtype={dt}")

mask_key = mask_candidates[0][1] if len(mask_candidates) else None

print("\n=== selected keys ===")
print("video_key =", video_key)
print("mask_key  =", mask_key)

assert video_key is not None, "video_key を推定できませんでした。候補一覧を見て手動で指定してください。"

# 念のため、選ばれた video_key のレンジを確認（ここが変なら video_key が間違い）
v0 = ex[video_key]
print(f"\n[check] {video_key}: dtype={v0.dtype}, min={v0.min()}, max={v0.max()}, shape={v0.shape}")

## 5. TFDS → PyTorch（動画シーケンス DataLoader）

In [ ]:
# 連続Tフレームを切り出す IterableDataset（正規化を堅牢化）
class TFDSVideoSeqIterable(IterableDataset):
    """
    yields:
      x:  [T,3,H,W] float32 in [-1,1]
      gt: [T,H,W]   int64 instance-id map (if mask_key exists) else None
    """
    def __init__(self, tfds_dataset, video_key, mask_key=None, T=24, img_size=128,
                 start_mode="random", seed=0):
        super().__init__()
        self.ds = tfds_dataset
        self.video_key = video_key
        self.mask_key = mask_key
        self.T = T
        self.img_size = img_size
        self.start_mode = start_mode
        self.rng = np.random.RandomState(seed)

    def _to_minus1_1(self, clip_np):
        """
        clip_np: [T,H,W,3] numpy
        return: torch float32 [T,3,H,W] in [-1,1]
        """
        # numpy -> torch
        x = torch.from_numpy(clip_np)

        # dtype/レンジに応じて正規化
        if x.dtype == torch.uint8:
            x = x.float() / 127.5 - 1.0  # 0..255 -> [-1,1]
        else:
            x = x.float()
            # 代表値でレンジ推定（軽く）
            mx = float(x.max())
            mn = float(x.min())
            # 0..1 なら [-1,1] へ
            if mn >= 0.0 and mx <= 1.0:
                x = x * 2.0 - 1.0
            # 0..255 っぽいなら [-1,1]
            elif mn >= 0.0 and mx <= 255.0:
                x = x / 127.5 - 1.0
            else:
                # それ以外は一旦クリップ（異常値対策）
                x = x.clamp(-1.0, 1.0)

        # [T,H,W,3] -> [T,3,H,W]
        x = x.permute(0, 3, 1, 2).contiguous()
        return x

    def __iter__(self):
        for ex in tfds.as_numpy(self.ds):
            video = ex[self.video_key]  # [Tv,H,W,3]
            Tv = video.shape[0]
            if Tv < self.T:
                continue

            s = int(self.rng.randint(0, Tv - self.T + 1)) if self.start_mode == "random" else 0
            clip = video[s:s+self.T]  # [T,H,W,3]

            x = self._to_minus1_1(clip)  # [T,3,H,W] in [-1,1]

            if self.img_size is not None and x.shape[-1] != self.img_size:
                x = F.interpolate(x, size=(self.img_size, self.img_size),
                                  mode="bilinear", align_corners=False)

            gt = None
            if self.mask_key is not None and self.mask_key in ex:
                m = ex[self.mask_key]
                if m.ndim == 4 and m.shape[-1] == 1:
                    m = m[..., 0]
                m = torch.from_numpy(m[s:s+self.T]).long()  # [T,H,W]
                if self.img_size is not None and m.shape[-1] != self.img_size:
                    m = m.unsqueeze(1).float()
                    m = F.interpolate(m, size=(self.img_size, self.img_size), mode="nearest")
                    m = m[:, 0].long()
                gt = m

            yield x, gt

# ---- 主要ハイパラ（まずはSAVi-S相当の軽め設定）----
IMG_SIZE = 128
T_TRAIN  = 12
T_EVAL   = 24
BATCH_SIZE = 8

train_seq = TFDSVideoSeqIterable(train_tfds, video_key, mask_key, T=T_TRAIN, img_size=IMG_SIZE, start_mode="random", seed=0)
train_loader = DataLoader(train_seq, batch_size=BATCH_SIZE, num_workers=0)

if val_tfds is not None:
    val_seq = TFDSVideoSeqIterable(val_tfds, video_key, mask_key, T=T_EVAL, img_size=IMG_SIZE, start_mode="random", seed=1)
    val_loader = DataLoader(val_seq, batch_size=4, num_workers=0)
else:
    val_loader = None

# 動作確認
x, gt = next(iter(train_loader))
print("x:", x.shape, x.dtype, x.min().item(), x.max().item(), "mean", x.mean().item())
print("gt:", None if gt is None else (gt.shape, gt.dtype))

## 6. 可視化（入力動画の確認）

In [ ]:
def show_video_grid(x, n_frames=8, title="input"):
    # x: [B,T,3,H,W] in [-1,1]
    x = x[0, :n_frames].detach().cpu()
    x = (x + 1) / 2
    fig, axes = plt.subplots(1, n_frames, figsize=(2*n_frames, 2))
    for i in range(n_frames):
        axes[i].imshow(x[i].permute(1,2,0).clamp(0,1))
        axes[i].axis("off")
        axes[i].set_title(str(i))
    fig.suptitle(title)
    plt.show()

x, _ = next(iter(train_loader))
show_video_grid(x, n_frames=min(8, x.shape[1]), title="MOVi-A clip")

## 7. SAVi のコア：Predict（slot transition）→ Correct（slot attention）

In [ ]:
class SlotAttention(nn.Module):
    """Slot Attention（slots_init を受け取れるように拡張）"""
    def __init__(self, num_slots=7, dim=64, iters=3, hidden_dim=128, eps=1e-8):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.dim = dim
        self.eps = eps

        self.norm_inputs = nn.LayerNorm(dim)
        self.norm_slots  = nn.LayerNorm(dim)
        self.norm_mlp    = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.gru = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, dim)
        )

        self.slots_mu = nn.Parameter(torch.zeros(1, 1, dim))
        self.slots_sigma = nn.Parameter(torch.ones(1, 1, dim))

    def init_slots(self, B, device=None):
        device = device or self.slots_mu.device
        mu = self.slots_mu.expand(B, self.num_slots, -1)
        sigma = self.slots_sigma.expand(B, self.num_slots, -1)
        return mu + sigma * torch.randn_like(mu, device=device)

    def forward(self, inputs, slots_init=None):
        # inputs: [B,N,D]
        B, N, D = inputs.shape
        inputs = self.norm_inputs(inputs)

        if slots_init is None:
            slots = self.init_slots(B, device=inputs.device)
        else:
            slots = slots_init

        k = self.to_k(inputs)
        v = self.to_v(inputs)

        for _ in range(self.iters):
            slots_prev = slots
            slots_norm = self.norm_slots(slots)
            q = self.to_q(slots_norm)

            attn_logits = torch.einsum("bnd,bkd->bnk", k, q) * (D ** -0.5)
            attn = F.softmax(attn_logits, dim=-1) + self.eps  # over slots

            attn_norm = attn / attn.sum(dim=1, keepdim=True)
            updates = torch.einsum("bnk,bnd->bkd", attn_norm, v)

            slots = self.gru(
                updates.reshape(B*self.num_slots, D),
                slots_prev.reshape(B*self.num_slots, D)
            ).reshape(B, self.num_slots, D)

            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots, attn  # slots [B,K,D], attn [B,N,K]


class SlotTransition(nn.Module):
    """SAVi の Predict ステップ：slot間相互作用（self-attn）+ MLP で次状態を予測"""
    def __init__(self, dim=64, num_heads=4, mlp_hidden=128):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.ReLU(inplace=True),
            nn.Linear(mlp_hidden, dim)
        )

    def forward(self, slots):
        # slots: [B,K,D]
        h = self.ln1(slots)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        slots = slots + attn_out
        slots = slots + self.mlp(self.ln2(slots))
        return slots

## 8. Encoder / Decoder

In [ ]:
class SoftPositionEmbed(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.dense = nn.Linear(4, hidden_size)

    def forward(self, x):
        B, C, H, W = x.shape
        device = x.device
        ys = torch.linspace(0., 1., H, device=device)
        xs = torch.linspace(0., 1., W, device=device)
        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        coords = torch.stack([xx, yy, 1.-xx, 1.-yy], dim=-1)  # [H,W,4]
        emb = self.dense(coords).permute(2,0,1).unsqueeze(0)  # [1,C,H,W]
        return x + emb

class CNNEncoder(nn.Module):
    def __init__(self, in_ch=3, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 5, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 5, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 5, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 5, padding=2), nn.ReLU(inplace=True),
        )
        self.pos = SoftPositionEmbed(hidden)
        self.ln = nn.LayerNorm(hidden)
        self.mlp = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
        )

    def forward(self, x):
        h = self.conv(x)
        h = self.pos(h)
        B, C, H, W = h.shape
        h = h.permute(0,2,3,1).reshape(B, H*W, C)
        h = self.ln(h)
        h = self.mlp(h)
        return h, (H, W)

class SpatialBroadcastDecoder(nn.Module):
    def __init__(self, slot_dim=64, hidden=64, out_size=128, init_res=8):
        super().__init__()
        self.out_size = out_size
        self.init_res = init_res
        self.pos = SoftPositionEmbed(slot_dim)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(slot_dim, hidden, 5, stride=2, padding=2, output_padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, hidden, 5, stride=2, padding=2, output_padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, hidden, 5, stride=2, padding=2, output_padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, hidden, 5, stride=2, padding=2, output_padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 5, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 4, 3, padding=1),
        )

    def forward(self, slots):
        B, K, D = slots.shape
        x = slots.reshape(B*K, D, 1, 1).expand(B*K, D, self.init_res, self.init_res)
        x = self.pos(x)
        out = self.deconv(x)  # [B*K,4,H,W]
        out = out.view(B, K, 4, self.out_size, self.out_size)
        rgb = out[:, :, :3]
        alpha = out[:, :, 3:4]
        masks = F.softmax(alpha, dim=1)          # over slots
        recon = (masks * rgb).sum(dim=1)
        return recon, masks, rgb

## 9. SAVi モデル（Predict→Correct をフレーム毎に回す）

In [ ]:
class SAVi(nn.Module):
    """SAVi-style Slot Attention for Video (reconstruction objective version)"""
    def __init__(self, num_slots=7, slot_dim=64, iters=3, trans_heads=4):
        super().__init__()
        self.encoder = CNNEncoder(in_ch=3, hidden=slot_dim)
        self.corrector = SlotAttention(num_slots=num_slots, dim=slot_dim, iters=iters, hidden_dim=128)
        self.transition = SlotTransition(dim=slot_dim, num_heads=trans_heads, mlp_hidden=2*slot_dim)
        self.decoder = SpatialBroadcastDecoder(slot_dim=slot_dim, hidden=64, out_size=IMG_SIZE, init_res=8)

    def forward(self, x):
        """
        x: [B,T,3,H,W] in [-1,1]
        returns dict of tensors with time dimension
        """
        B, T, C, H, W = x.shape
        slots = self.corrector.init_slots(B, device=x.device)

        recons = []
        masks  = []
        rgbs   = []
        slots_all = []
        attn_all = []

        for t in range(T):
            xt = x[:, t]
            feats, _ = self.encoder(xt)  # [B,N,D]

            # Predict step (except t=0: no previous info to predict from)
            if t > 0:
                slots = self.transition(slots)

            # Correct step (Slot Attention, initialized by predicted slots)
            slots, attn = self.corrector(feats, slots_init=slots)

            # Decode
            recon, m, rgb = self.decoder(slots)

            recons.append(recon)
            masks.append(m)
            rgbs.append(rgb)
            slots_all.append(slots)
            attn_all.append(attn)

        out = {
            "recon": torch.stack(recons, dim=1),        # [B,T,3,H,W]
            "masks": torch.stack(masks, dim=1),         # [B,T,K,1,H,W]
            "rgb_slots": torch.stack(rgbs, dim=1),      # [B,T,K,3,H,W]
            "slots": torch.stack(slots_all, dim=1),     # [B,T,K,D]
            "attn": torch.stack(attn_all, dim=1),       # [B,T,N,K]
        }
        return out

## 10. 学習準備（最小：reconstruction loss）

In [ ]:
# =========================
# 10. 学習準備（baseline-ready + collapse対策）
# warmup + cosine decay / AMP / collapse diagnostics / ckpt / smoke test
# =========================

import os, math, time
import numpy as np
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment

import torch
import torch.nn.functional as F

# -------------------------
# Hyperparameters（★collapse対策を反映）
# -------------------------
# モデル
NUM_SLOTS = 7
SLOT_DIM  = 64
ITERS     = 3

# ★最適化（collapse対策）
BASE_LR       = 2e-4     # 4e-4 -> 2e-4
WEIGHT_DECAY  = 0.0
GRAD_CLIP     = 0.5      # 1.0 -> 0.5

# 学習ステップ（ベースライン）
MAX_STEPS     = 80_000
WARMUP_STEPS  = 20_000   # 10k -> 20k
MIN_LR_RATIO  = 0.05

# ログ/評価/可視化/保存
LOG_EVERY     = 50
EVAL_EVERY    = 5_000
VIS_EVERY     = 10_000
CKPT_EVERY    = 10_000
VAL_BATCHES   = 20

# AMP（GPUならON推奨）
USE_AMP = (device.type == "cuda")

# 速度系
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


# -------------------------
# Model / Optimizer / Scheduler
# -------------------------
model = SAVi(num_slots=NUM_SLOTS, slot_dim=SLOT_DIM, iters=ITERS, trans_heads=4).to(device)

opt = torch.optim.Adam(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(step: int):
    """linear warmup -> cosine decay to MIN_LR_RATIO"""
    if step < 1:
        return 0.0
    if step < WARMUP_STEPS:
        return step / float(WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / float(max(1, MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_lambda)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


# -------------------------
# Loss / helpers
# -------------------------
def recon_loss(recon, x):
    return F.mse_loss(recon, x)

def mse_to_psnr(mse):
    return 20.0 * math.log10(2.0) - 10.0 * math.log10(max(mse, 1e-12))


# -------------------------
# Collapse diagnostics
# -------------------------
@torch.no_grad()
def compute_slot_diagnostics(out):
    masks = out["masks"].squeeze(3)  # [B,T,K,H,W]
    B, T, K, H, W = masks.shape

    p = masks.clamp(1e-8, 1.0)
    pix_ent = -(p * p.log()).sum(dim=2).mean().item()

    usage = masks.mean(dim=(-1, -2))          # [B,T,K]
    usage_mean = usage.mean(dim=(0, 1))       # [K]
    usage_max = float(usage_mean.max().item())
    usage_min = float(usage_mean.min().item())

    u = usage_mean.clamp(1e-8, 1.0)
    usage_ent = float((-(u * u.log()).sum()).item())

    slots = out["slots"]                      # [B,T,K,D]
    s = slots.reshape(-1, K, slots.shape[-1]) # [B*T,K,D]
    s = F.normalize(s, dim=-1)
    sim = torch.matmul(s, s.transpose(1, 2))  # [B*T,K,K]

    if K <= 1:
        offdiag = 0.0
    else:
        eye = torch.eye(K, dtype=torch.bool, device=sim.device).unsqueeze(0)
        eye = eye.expand(sim.shape[0], -1, -1)
        offdiag = sim.masked_select(~eye).mean().item()

    return {
        "pix_entropy": float(pix_ent),
        "usage_max": float(usage_max),
        "usage_min": float(usage_min),
        "usage_entropy": float(usage_ent),
        "slot_cos_offdiag": float(offdiag),
    }


# -------------------------
# Checkpoint
# -------------------------
CKPT_DIR = "./checkpoints_savi_movia"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_ckpt(step, extra=None):
    path = os.path.join(CKPT_DIR, f"ckpt_step{step:07d}.pt")
    payload = {
        "step": step,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if USE_AMP else None,
        "extra": extra,
        "hparams": {
            "NUM_SLOTS": NUM_SLOTS, "SLOT_DIM": SLOT_DIM, "ITERS": ITERS,
            "BASE_LR": BASE_LR, "MAX_STEPS": MAX_STEPS,
            "WARMUP_STEPS": WARMUP_STEPS, "MIN_LR_RATIO": MIN_LR_RATIO,
            "GRAD_CLIP": GRAD_CLIP
        }
    }
    torch.save(payload, path)
    return path

def load_ckpt(path):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["opt"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if USE_AMP and ckpt.get("scaler", None) is not None:
        scaler.load_state_dict(ckpt["scaler"])
    return int(ckpt["step"])


# -------------------------
# Quick smoke test（train_loaderの返り値が何要素でもOK）
# -------------------------
@torch.no_grad()
def eval_batch(model, x):
    model.eval()
    out = model(x)
    loss = recon_loss(out["recon"], x).item()
    diag = compute_slot_diagnostics(out)
    return loss, out, diag

def _get_x_from_batch(batch):
    if isinstance(batch, dict):
        for k in ["x", "video", "frames", "images"]:
            if k in batch:
                return batch[k]
        return next(iter(batch.values()))
    if isinstance(batch, (list, tuple)):
        return batch[0]
    return batch

batch = next(iter(train_loader))
x = _get_x_from_batch(batch).to(device)

loss0, out0, diag0 = eval_batch(model, x)
print("init recon mse:", float(loss0), "psnr:", float(mse_to_psnr(loss0)))
print("init diag:", diag0)
print(f"[HPARAMS] BASE_LR={BASE_LR} WARMUP_STEPS={WARMUP_STEPS} GRAD_CLIP={GRAD_CLIP}")

## 11. 可視化（動画 recon / slot masks）

In [ ]:
# --- 可視化（動画 recon / slot / mask）---

import matplotlib.pyplot as plt

@torch.no_grad()
def visualize_savi(out, x, n_frames=8, slot_vis=4):
    """
    out: model(x) の出力 dict
      out["recon"] : [B,T,3,H,W]
      out["masks"] : [B,T,K,1,H,W]
      out["recons"]: [B,T,K,3,H,W]   (もし無ければ maskだけ表示)
    x:   入力 [B,T,3,H,W]
    """
    model_was_training = model.training
    model.eval()

    B, T, C, H, W = x.shape
    n_frames = min(n_frames, T)

    # 入力と recon
    x_np = (x[0, :n_frames].detach().cpu().permute(0,2,3,1).numpy() + 1) / 2.0
    recon = out["recon"][0, :n_frames].detach().cpu().permute(0,2,3,1).numpy()
    recon = (recon + 1) / 2.0

    fig, axes = plt.subplots(2, n_frames, figsize=(2*n_frames, 4))
    for t in range(n_frames):
        axes[0, t].imshow(x_np[t])
        axes[0, t].axis("off")
        if t == 0: axes[0, t].set_title("input")

        axes[1, t].imshow(np.clip(recon[t], 0, 1))
        axes[1, t].axis("off")
        if t == 0: axes[1, t].set_title("recon")
    plt.show()

    # masks（slotごと）
    masks = out["masks"][0, :n_frames]  # [T,K,1,H,W]
    K = masks.shape[1]
    slot_vis = min(slot_vis, K)

    fig, axes = plt.subplots(slot_vis, n_frames, figsize=(2*n_frames, 2*slot_vis))
    if slot_vis == 1:
        axes = np.expand_dims(axes, 0)

    for s in range(slot_vis):
        for t in range(n_frames):
            m = masks[t, s, 0].detach().cpu().numpy()  # [H,W]
            axes[s, t].imshow(m, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            axes[s, t].axis("off")
            if t == 0:
                axes[s, t].set_title(f"slot {s}")
    plt.show()

    # per-slot recon（存在する場合）
    if "recons" in out:
        recons = out["recons"][0, :n_frames]  # [T,K,3,H,W]
        fig, axes = plt.subplots(slot_vis, n_frames, figsize=(2*n_frames, 2*slot_vis))
        if slot_vis == 1:
            axes = np.expand_dims(axes, 0)
        for s in range(slot_vis):
            for t in range(n_frames):
                r = recons[t, s].detach().cpu().permute(1,2,0).numpy()
                r = (r + 1) / 2.0
                axes[s, t].imshow(np.clip(r, 0, 1))
                axes[s, t].axis("off")
                if t == 0:
                    axes[s, t].set_title(f"recon slot {s}")
        plt.show()

    if model_was_training:
        model.train()


# --- quick smoke test（ここが今回落ちていた部分の修正）---
# train_loader が (x,gt) でも (x,gt,meta...) でも動くようにする
batch = next(iter(train_loader))
x = batch[0].to(device)   # 先頭が入力
# gt は必要なら batch[1] を使う（ここでは可視化だけなので不要）

# eval_batch が (loss, out, diag) を返す前提で受け取る
loss, out, diag = eval_batch(model, x)

print("init recon loss (mse):", float(loss))
print("init diag:", diag)

visualize_savi(out, x, n_frames=min(8, x.shape[1]), slot_vis=4)

## 12. 学習ループ（SAVi：Predict→Correct を時間方向に回して学習）

In [ ]:
# ===========================
# Cell12: FULL integrated training loop (one-cell, grad-safe)
# - optimizer alias fix
# - ALWAYS train with model(x) forward (grad-enabled)
# - loss is guaranteed to require grad
# - torch.amp (no deprecation)
# - history logging (train + optional val)
# - train metrics: loss, recon_mse, recon_l1, psnr (+ optional fg_ari/gt_miou, tracking)
# - optional visualization, checkpoint, history save
# ===========================

import os, json, time
from collections import defaultdict
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# optional scipy for Hungarian (FG-ARI/GT-mIoU matching)
try:
    from scipy.optimize import linear_sum_assignment
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# 0) Normalize optimizer name
# ----------------------------
if "optimizer" not in globals():
    for _cand in ["optim", "opt", "adam", "optimizer_", "optm", "my_optimizer"]:
        if _cand in globals():
            optimizer = globals()[_cand]
            print(f"[INFO] optimizer was not defined; using `{_cand}` as optimizer.")
            break
if "optimizer" not in globals():
    raise NameError(
        "optimizer が未定義です。ノートブック内の最適化器変数名（例: optim / opt）を確認し、"
        "このセル先頭の候補リストに追加するか、optimizer = <your_optimizer> を追加してください。"
    )

# ----------------------------
# 1) SETTINGS (edit as needed)
# ----------------------------
MAX_STEPS        = 80000   # 必要に応じて変更

LOG_EVERY        = 20
TRAIN_MET_EVERY  = 200      # train FG-ARI/GT-mIoU（GTあり＆scipyあり）
TRACK_MET_EVERY  = 500      # train tracking（compute_id_swap_statsがある場合）
VIS_EVERY        = 1000     # 可視化（visualize_saviがある場合）
EVAL_EVERY       = 1000     # val評価（evaluate_baseline & val_loaderがある場合）
SAVE_CKPT_EVERY  = 2000
SAVE_HIST_EVERY  = 2000

FIG_DIR   = "./figs_learning_curves"
CKPT_DIR  = "./checkpoints"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

GRAD_CLIP_NORM = 1.0
USE_AMP = (device.type == "cuda")

FIXED_BS     = 1
FIXED_FRAMES = 8
FIXED_SLOTS  = 4

# ----------------------------
# 2) HISTORY logger
# ----------------------------
history = defaultdict(list)

def log_history(step: int, epoch: int = None, split: str = "train", **metrics):
    history["step"].append(int(step))
    history["epoch"].append(None if epoch is None else int(epoch))
    history["split"].append(str(split))
    history["time"].append(time.time())
    for k, v in metrics.items():
        if hasattr(v, "detach"):
            v = v.detach().float().mean().item()
        try:
            v = float(v)
        except Exception:
            pass
        history[k].append(v)

def save_history_json(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({k: list(v) for k, v in history.items()}, f, indent=2)
    print("[OK] saved history:", path)

# ----------------------------
# 3) Metrics utils
# ----------------------------
@torch.no_grad()
def _hard_ids_from_soft_masks(m_tk1hw: torch.Tensor) -> torch.Tensor:
    # [T,K,1,H,W] -> [T,H,W]
    m = m_tk1hw[:, :, 0]
    return torch.argmax(m, dim=1).long()

def _psnr_from_mse(mse: torch.Tensor, max_val: float = 2.0) -> torch.Tensor:
    mse = torch.clamp(mse, min=1e-10)
    return 20.0 * torch.log10(torch.tensor(max_val, device=mse.device)) - 10.0 * torch.log10(mse)

def _ari_1d(true_labels: np.ndarray, pred_labels: np.ndarray) -> float:
    true_u, true_inv = np.unique(true_labels, return_inverse=True)
    pred_u, pred_inv = np.unique(pred_labels, return_inverse=True)
    cont = np.zeros((true_u.size, pred_u.size), dtype=np.int64)
    np.add.at(cont, (true_inv, pred_inv), 1)

    def comb2(x): return (x * (x - 1)) // 2
    nij = cont
    a = nij.sum(axis=1)
    b = nij.sum(axis=0)
    n = nij.sum()
    if n <= 1:
        return 1.0
    index = comb2(nij).sum()
    sum_a = comb2(a).sum()
    sum_b = comb2(b).sum()
    expected = sum_a * sum_b / comb2(n)
    max_index = 0.5 * (sum_a + sum_b)
    denom = max_index - expected
    if denom == 0:
        return 1.0
    return float((index - expected) / denom)

@torch.no_grad()
def compute_fg_ari_and_gt_miou(pred_masks_tk1hw: torch.Tensor,
                              gt_thw: torch.Tensor,
                              max_eval_frames: int = 8,
                              downsample: int = 2,
                              min_gt_area: int = 25):
    if not _HAS_SCIPY:
        return None, None

    device = pred_masks_tk1hw.device
    T, K, _, H, W = pred_masks_tk1hw.shape
    T_use = min(T, max_eval_frames)

    pm = pred_masks_tk1hw[:T_use]
    gt = gt_thw[:T_use].to(device).long()

    if downsample and downsample > 1:
        newH, newW = H // downsample, W // downsample
        gt = F.interpolate(gt.unsqueeze(1).float(), size=(newH, newW), mode="nearest")[:,0].long()
        pm = F.interpolate(pm, size=(newH, newW), mode="nearest")

    pred_ids = _hard_ids_from_soft_masks(pm)  # [T,h,w]

    # FG-ARI
    fg_aris = []
    for t in range(T_use):
        gt_t = gt[t].detach().cpu().numpy()
        pr_t = pred_ids[t].detach().cpu().numpy()
        fg = gt_t > 0
        if fg.sum() >= 2:
            fg_aris.append(_ari_1d(gt_t[fg], pr_t[fg]))
    fg_ari = float(np.mean(fg_aris)) if len(fg_aris) else 0.0

    # GT-mIoU (Hungarian per frame)
    gt_mious = []
    for t in range(T_use):
        gt_t = gt[t]
        pr_t = pred_ids[t]
        inst_ids = torch.unique(gt_t)
        inst_ids = inst_ids[inst_ids != 0]
        if inst_ids.numel() == 0:
            continue
        gt_masks = []
        for gid in inst_ids.tolist():
            m = (gt_t == gid)
            if int(m.sum()) >= min_gt_area:
                gt_masks.append(m)
        if len(gt_masks) == 0:
            continue
        gt_masks = torch.stack(gt_masks, dim=0)  # [M,h,w]
        pred_masks_hard = torch.stack([(pr_t == k) for k in range(K)], dim=0)  # [K,h,w]

        inter = (gt_masks[:,None] & pred_masks_hard[None,:]).flatten(2).sum(-1).float()
        union = (gt_masks[:,None] | pred_masks_hard[None,:]).flatten(2).sum(-1).float()
        iou = inter / (union + 1e-6)  # [M,K]

        iou_cpu = iou.detach().cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(1.0 - iou_cpu)
        matched = iou_cpu[row_ind, col_ind]
        gt_mious.append(float(matched.mean()) if matched.size else 0.0)

    gt_miou = float(np.mean(gt_mious)) if len(gt_mious) else 0.0
    return fg_ari, gt_miou

# ----------------------------
# 4) Checkpoint helper
# ----------------------------
def save_checkpoint(path, model, optimizer=None, step=None, extra=None):
    ckpt = {
        "model_state": model.state_dict(),
        "step": step,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if optimizer is not None:
        ckpt["optim_state"] = optimizer.state_dict()
    if extra is not None:
        ckpt["extra"] = extra
    torch.save(ckpt, path)
    print(f"[OK] saved checkpoint -> {path}")

# ----------------------------
# 5) Helper: normalize model output to dict
# ----------------------------
def _to_out_dict(out_raw):
    """
    model(x) の返り値が dict / (dict, ...) / (loss, dict, ...) 等でも
    dict部分を取り出す。無ければそのまま返す（後でエラー）。
    """
    if isinstance(out_raw, dict):
        return out_raw
    if isinstance(out_raw, (tuple, list)):
        d = next((z for z in out_raw if isinstance(z, dict)), None)
        return d if d is not None else out_raw
    return out_raw

# ----------------------------
# 6) Fixed batch for visualization
# ----------------------------
fixed_x, fixed_gt = next(iter(train_loader))
fixed_x = fixed_x[:FIXED_BS].to(device)
fixed_gt = None if fixed_gt is None else fixed_gt[:FIXED_BS].to(device)

# ----------------------------
# 7) Training loop
# ----------------------------
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

model.train()
global_step = 0
pbar = tqdm(total=MAX_STEPS, desc="train", leave=True)

# ensure grad enabled
torch.set_grad_enabled(True)

while global_step < MAX_STEPS:
    for x, gt in train_loader:
        if global_step >= MAX_STEPS:
            break

        x = x.to(device, non_blocking=True)
        gt = None if gt is None else gt.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # -------- forward (ALWAYS grad-enabled via model(x)) --------
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            out_raw = model(x)
            out = _to_out_dict(out_raw)

            if not (isinstance(out, dict) and ("recon" in out)):
                raise RuntimeError(
                    f"Model output must be dict with key 'recon'. got type={type(out)} "
                    f"keys={list(out.keys()) if isinstance(out, dict) else None}"
                )

            recon = out["recon"]
            recon_mse = ((recon - x) ** 2).mean()
            recon_l1  = (recon - x).abs().mean()
            psnr      = _psnr_from_mse(recon_mse, max_val=2.0)

            # -------- loss selection (prefer out['loss'] if it has grad) --------
            loss = None
            for k in ["loss", "total_loss", "train_loss", "recon_loss"]:
                if k in out and torch.is_tensor(out[k]) and out[k].requires_grad:
                    loss = out[k]
                    break
            if loss is None:
                # default: use differentiable recon loss
                loss = recon_mse

        # safety: loss must require grad
        if (not torch.is_tensor(loss)) or (not loss.requires_grad):
            # This should not happen now, but keep guard
            raise RuntimeError(
                f"loss does not require grad. type={type(loss)}, requires_grad={getattr(loss,'requires_grad',None)}. "
                "This indicates model forward is not producing a differentiable loss."
            )

        # -------- backward --------
        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        # -------- history (every step) --------
        log_history(global_step, epoch=None, split="train",
                    loss=float(loss.detach().cpu()),
                    recon_mse=float(recon_mse.detach().cpu()),
                    recon_l1=float(recon_l1.detach().cpu()),
                    psnr=float(psnr.detach().cpu()))

        # -------- train FG-ARI / GT-mIoU（GTあり＆masksあり）--------
        if (global_step % TRAIN_MET_EVERY == 0) and (gt is not None) and isinstance(out, dict) and ("masks" in out):
            try:
                fg_ari, gt_miou = compute_fg_ari_and_gt_miou(out["masks"][0], gt[0],
                                                            max_eval_frames=min(8, out["masks"].shape[1]),
                                                            downsample=2,
                                                            min_gt_area=25)
                if fg_ari is not None:
                    log_history(global_step, epoch=None, split="train", fg_ari=fg_ari, gt_miou=gt_miou)
            except Exception as e:
                print("[TRAIN MET WARNING]", type(e).__name__, e)

        # -------- tracking metrics（関数がある＆masksあり）--------
        if (global_step % TRACK_MET_EVERY == 0) and ("compute_id_swap_stats" in globals()) and isinstance(out, dict) and ("masks" in out):
            try:
                stats = compute_id_swap_stats(out["masks"][0])
                log_history(global_step, epoch=None, split="train",
                            id_swap_mean=stats.get("id_swap_mean", 0.0),
                            matched_iou_mean=stats.get("matched_iou_mean", 0.0),
                            valid_frac_mean=stats.get("valid_frac_mean", 0.0))
            except Exception as e:
                print("[TRACK MET WARNING]", type(e).__name__, e)

        # -------- progress --------
        if global_step % LOG_EVERY == 0:
            pbar.set_postfix({
                "loss": f"{float(loss.detach().cpu()):.4f}",
                "mse": f"{float(recon_mse.detach().cpu()):.4f}",
                "psnr": f"{float(psnr.detach().cpu()):.2f}"
            })

        # -------- visualization --------
        if (global_step % VIS_EVERY == 0) and ("visualize_savi" in globals()):
            model.eval()
            with torch.no_grad():
                out_vis = _to_out_dict(model(fixed_x))
            try:
                visualize_savi(out_vis, fixed_x, n_frames=min(FIXED_FRAMES, fixed_x.shape[1]), slot_vis=FIXED_SLOTS)
            except Exception as e:
                print("[VIS WARNING]", type(e).__name__, e)
            model.train()

        # -------- val baseline evaluation --------
        if (global_step % EVAL_EVERY == 0) and ("evaluate_baseline" in globals()) and ("val_loader" in globals()) and (val_loader is not None):
            model.eval()
            with torch.no_grad():
                res = evaluate_baseline(model, val_loader, n_batches=50)
            safe_res = {}
            for k, v in res.items():
                try:
                    safe_res[k] = float(v)
                except Exception:
                    pass
            log_history(global_step, epoch=None, split="val", **safe_res)
            print("[VAL]", safe_res)
            model.train()

        # -------- checkpoint saving --------
        if global_step % SAVE_CKPT_EVERY == 0:
            ckpt_path = os.path.join(CKPT_DIR, f"savi_step{global_step}.pt")
            save_checkpoint(ckpt_path, model=model, optimizer=optimizer, step=global_step, extra={"USE_AMP": USE_AMP})
            save_checkpoint(os.path.join(CKPT_DIR, "savi_latest.pt"), model=model, optimizer=optimizer, step=global_step, extra=None)

        # -------- history saving --------
        if global_step % SAVE_HIST_EVERY == 0:
            save_history_json(os.path.join(FIG_DIR, "history.json"))

        global_step += 1
        pbar.update(1)

pbar.close()
print("[DONE] training finished. steps =", global_step)

# final save
save_history_json(os.path.join(FIG_DIR, "history.json"))
save_checkpoint(os.path.join(CKPT_DIR, "savi_latest.pt"), model=model, optimizer=optimizer, step=global_step, extra=None)

## 13. Hungarian追跡・ID swap率（SAVi出力に対するベースライン評価）
ここでは **SAViのスロットmask**（`out["masks"]`）に対して、  
フレーム間で **Hungarian（最大IoU）対応付け**を行い、スロットの「ID（slot index）」が時間方向にどれだけ安定しているかを **ID swap率** として測定。

- `ID swap率`：各フレームで得られる最適対応（permutation）が **identity（prev slot k → curr slot k）からどれだけズレたか**の割合  
  - **小さいほど**「同じスロットが同じ物体を追い続けている」傾向（=SAViが狙う性質）
- 併せて `matched IoU`（対応付け後の平均IoU）も出力


In [ ]:
# --- Hungarian追跡ユーティリティ ---
from scipy.optimize import linear_sum_assignment

@torch.no_grad()
def disjoint_binary_masks(masks_tk1hw: torch.Tensor) -> torch.Tensor:
    '''
    masks_tk1hw: [T,K,1,H,W] (soft, sum over K = 1 per pixel)
    returns: bin_masks [T,K,H,W] (argmax-based disjoint binary masks)
    '''
    assert masks_tk1hw.ndim == 5
    T, K, _, H, W = masks_tk1hw.shape
    probs = masks_tk1hw.squeeze(2)  # [T,K,H,W]
    labels = probs.argmax(dim=1)    # [T,H,W]
    oh = F.one_hot(labels, num_classes=K).permute(0,3,1,2).to(probs.dtype)  # [T,K,H,W]
    return oh

@torch.no_grad()
def iou_matrix(bin_prev_khw: torch.Tensor, bin_curr_khw: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    '''
    bin_prev_khw, bin_curr_khw: [K,H,W] binary (0/1 float ok)
    returns: IoU [K,K]
    '''
    K, H, W = bin_prev_khw.shape
    a = bin_prev_khw.reshape(K, -1)  # [K,HW]
    b = bin_curr_khw.reshape(K, -1)  # [K,HW]
    inter = a @ b.t()  # [K,K]
    area_a = a.sum(dim=1, keepdim=True)  # [K,1]
    area_b = b.sum(dim=1, keepdim=True).t()  # [1,K]
    union = area_a + area_b - inter
    return inter / (union + eps)

@torch.no_grad()
def hungarian_match_iou(iou_kK: torch.Tensor) -> np.ndarray:
    '''
    iou_kK: [K,K] torch
    returns perm: np.ndarray [K] where perm[prev_idx] = curr_idx
    '''
    iou_np = iou_kK.detach().cpu().numpy()
    row_ind, col_ind = linear_sum_assignment(-iou_np)  # maximize IoU
    perm = np.full((iou_np.shape[0],), -1, dtype=np.int64)
    perm[row_ind] = col_ind
    return perm

@torch.no_grad()
def compute_id_swap_stats(masks_tk1hw: torch.Tensor, min_area: float = 10.0):
    '''
    masks_tk1hw: [T,K,1,H,W] for a single sample
    min_area: pixels below this area treated as 'empty slot' and excluded from swap denominator
    returns dict with swap_rate, matched_iou
    '''
    T, K, _, H, W = masks_tk1hw.shape
    bin_masks = disjoint_binary_masks(masks_tk1hw)  # [T,K,H,W]

    swap_rates = []
    matched_ious = []

    for t in range(1, T):
        prev = bin_masks[t-1]  # [K,H,W]
        curr = bin_masks[t]

        area_prev = prev.reshape(K, -1).sum(dim=1)  # [K]
        valid_prev = (area_prev > min_area)

        iou = iou_matrix(prev, curr)  # [K,K]
        perm = hungarian_match_iou(iou)  # [K]

        idx = np.arange(K, dtype=np.int64)
        valid_prev_np = valid_prev.detach().cpu().numpy().astype(bool)
        denom = max(int(valid_prev_np.sum()), 1)
        swap = (perm[valid_prev_np] != idx[valid_prev_np]).sum() / denom

        matched = iou.detach().cpu().numpy()[idx, perm]
        miou = float(matched[valid_prev_np].mean()) if valid_prev_np.sum() > 0 else float(matched.mean())

        swap_rates.append(float(swap))
        matched_ious.append(float(miou))

    return {
        "swap_rate_mean": float(np.mean(swap_rates)) if swap_rates else 0.0,
        "swap_rate_per_t": swap_rates,
        "matched_iou_mean": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "matched_iou_per_t": matched_ious,
    }


## 14. サンプルで追跡指標を確認（デバッグ用）
学習後（ある程度 recon が安定した後）に実行。

In [ ]:
# --- 1サンプルで swap率・matched IoU を確認 ---
model.eval()
x, _ = next(iter(val_loader))
x = x[:1].to(device)  # 1サンプル
with torch.no_grad():
    out = model(x)

stats = compute_id_swap_stats(out["masks"][0], min_area=10.0)
print("ID swap率(mean):", stats["swap_rate_mean"])
print("matched IoU(mean):", stats["matched_iou_mean"])
print("per-frame swap:", [round(s, 3) for s in stats["swap_rate_per_t"][:10]], "...")
print("per-frame mIoU:", [round(s, 3) for s in stats["matched_iou_per_t"][:10]], "...")


## 15. バリデーション全体でベースライン評価（ID swap率）
少数バッチでサクッと平均を出力（重い場合は `N_EVAL_BATCHES` を下げる）。

In [ ]:
@torch.no_grad()
def evaluate_tracking_metrics(model, loader, n_batches=20, min_area=10.0):
    model.eval()
    swap_means = []
    miou_means = []
    for i, (x, _) in enumerate(loader):
        if i >= n_batches:
            break
        x = x.to(device)
        out = model(x)
        for b in range(out["masks"].shape[0]):
            s = compute_id_swap_stats(out["masks"][b], min_area=min_area)
            swap_means.append(s["swap_rate_mean"])
            miou_means.append(s["matched_iou_mean"])
    return float(np.mean(swap_means)), float(np.mean(miou_means))

N_EVAL_BATCHES = 20
swap_mean, miou_mean = evaluate_tracking_metrics(model, val_loader, n_batches=N_EVAL_BATCHES, min_area=10.0)
print(f"[Eval] batches={N_EVAL_BATCHES} | ID swap率(mean)={swap_mean:.4f} | matched IoU(mean)={miou_mean:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def _to01_same_device(x: torch.Tensor) -> torch.Tensor:
    """x: torch tensor on ANY device, [-1,1] or [0,1] -> [0,1] (same device)"""
    x = x.detach().float()
    if x.numel() > 0 and x.min() < -0.1:   # assume [-1,1]
        x = (x + 1) / 2
    return x.clamp(0, 1)

def _to_numpy_img(x: torch.Tensor) -> np.ndarray:
    """x: [3,H,W] or [H,W,3] torch on ANY device -> numpy [H,W,3] in [0,1]"""
    if x.ndim == 3 and x.shape[0] == 3:
        x = x.permute(1,2,0)
    return x.detach().cpu().numpy()

def _to_numpy_gray(x: torch.Tensor) -> np.ndarray:
    """x: [H,W] torch -> numpy [H,W]"""
    return x.detach().cpu().numpy()

@torch.no_grad()
def visualize_batch(out, x, frames=8, slots=4, title_prefix="", show_slot_recon=True):
    """
    x:   [B,T,3,H,W] on GPU (推奨)
    out: dict with keys:
         - "recon": [B,T,3,H,W]  (GPU)
         - "masks": [B,T,K,1,H,W] (GPU)
    計算はGPUのまま行い、描画直前のみcpuへ移す
    """
    # --- 同一deviceで扱う ---
    device = x.device
    recon = out["recon"].to(device)
    masks = out["masks"].to(device)

    B, T, C, H, W = x.shape
    frames = min(frames, T)
    K = masks.shape[2]
    slots = min(slots, K)

    x01 = _to01_same_device(x[0, :frames])        # [F,3,H,W] GPU
    r01 = _to01_same_device(recon[0, :frames])    # [F,3,H,W] GPU
    m   = masks[0, :frames]                       # [F,K,1,H,W] GPU

    # --- input vs recon ---
    fig, axes = plt.subplots(2, frames, figsize=(2*frames, 4))
    for t in range(frames):
        axes[0, t].imshow(_to_numpy_img(x01[t]))
        axes[0, t].set_title(f"{title_prefix}in {t}")
        axes[0, t].axis("off")

        axes[1, t].imshow(_to_numpy_img(r01[t]))
        axes[1, t].set_title(f"{title_prefix}recon {t}")
        axes[1, t].axis("off")
    plt.tight_layout()
    plt.show()

    # --- masks ---
    fig, axes = plt.subplots(slots, frames, figsize=(2*frames, 2*slots))
    if slots == 1:
        axes = np.expand_dims(axes, 0)
    for s in range(slots):
        for t in range(frames):
            mm = m[t, s, 0]  # [H,W] GPU
            axes[s, t].imshow(_to_numpy_gray(mm), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            axes[s, t].axis("off")
            if t == 0:
                axes[s, t].set_title(f"{title_prefix}slot {s}")
    plt.tight_layout()
    plt.show()

    # --- slot recon (GPUで計算してから描画) ---
    if show_slot_recon:
        fig, axes = plt.subplots(slots, frames, figsize=(2*frames, 2*slots))
        if slots == 1:
            axes = np.expand_dims(axes, 0)
        for s in range(slots):
            for t in range(frames):
                mm = m[t, s]        # [1,H,W] GPU
                sr = r01[t] * mm    # [3,H,W] GPU（←ここがGPU統一）
                axes[s, t].imshow(_to_numpy_img(sr))
                axes[s, t].axis("off")
                if t == 0:
                    axes[s, t].set_title(f"{title_prefix}slot-recon {s}")
        plt.tight_layout()
        plt.show()

    # --- デバッグ（GPU上のまま統計） ---
    print("[debug] x(min/max/mean):", float(x.min()), float(x.max()), float(x.mean()))
    print("[debug] recon(min/max/mean):", float(recon.min()), float(recon.max()), float(recon.mean()))
    print("[debug] masks(min/max/mean):", float(masks.min()), float(masks.max()), float(masks.mean()))

In [ ]:
# --- 推論用に1バッチ取り出し ---
x, gt = next(iter(val_loader if val_loader is not None else train_loader))
x = x.to(device)


model.eval()
with torch.no_grad():
    out = model(x)  # out["recon"], out["masks"]

visualize_batch(out, x, frames=8, slots=4, title_prefix="")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

@torch.no_grad()
def masks_to_hard_ids(m_soft: torch.Tensor) -> torch.Tensor:
    """
    m_soft: [K,H,W] or [K,1,H,W] soft masks (sum over K = 1)
    return: [H,W] hard assignment id map (0..K-1)
    """
    if m_soft.ndim == 4:
        m_soft = m_soft[:,0]  # [K,H,W]
    return torch.argmax(m_soft, dim=0)  # [H,W]

@torch.no_grad()
def iou_matrix_from_hard_ids(ids_a: torch.Tensor, ids_b: torch.Tensor, K: int) -> torch.Tensor:
    """
    ids_a, ids_b: [H,W] with values in 0..K-1 (hard assignment)
    return: [K,K] IoU matrix (GPU)
    """
    # one-hot: [K,H,W]
    a = torch.nn.functional.one_hot(ids_a.long(), num_classes=K).permute(2,0,1).bool()
    b = torch.nn.functional.one_hot(ids_b.long(), num_classes=K).permute(2,0,1).bool()

    # intersection/union
    inter = (a[:,None] & b[None,:]).flatten(2).sum(-1).float()  # [K,K]
    union = (a[:,None] | b[None,:]).flatten(2).sum(-1).float()  # [K,K]
    iou = inter / (union + 1e-6)
    return iou  # [K,K]

@torch.no_grad()
def hungarian_match_iou(iou: torch.Tensor):
    """
    iou: [K,K] torch tensor (GPU or CPU)
    return: perm (list length K) such that slot_b[perm[j]] matches slot_a[j]
    """
    iou_cpu = iou.detach().cpu().numpy()
    # maximize IoU => minimize (1 - IoU)
    row_ind, col_ind = linear_sum_assignment(1.0 - iou_cpu)
    perm = col_ind.tolist()  # for each row j, matched column is perm[j]
    return perm, iou_cpu[row_ind, col_ind].mean().item()

@torch.no_grad()
def track_slots_by_iou(masks_BT: torch.Tensor):
    """
    masks_BT: [T,K,1,H,W] or [T,K,H,W] (on GPU 推奨)
    return:
      perm_list[t]: list length K, mapping from reference order (t=0 slots)
                   to current frame slots
    """
    if masks_BT.ndim == 5:
        masks_BT = masks_BT[:,:,0]  # [T,K,H,W]
    T, K, H, W = masks_BT.shape
    device = masks_BT.device

    # reference: t=0 hard ids
    ids_ref = masks_to_hard_ids(masks_BT[0]).to(device)

    perm_list = []
    prev_ids = ids_ref
    prev_perm = list(range(K))
    perm_list.append(prev_perm)

    for t in range(1, T):
        ids_t = masks_to_hard_ids(masks_BT[t]).to(device)
        iou = iou_matrix_from_hard_ids(prev_ids, ids_t, K)  # [K,K] GPU
        perm, _ = hungarian_match_iou(iou)
        # update
        perm_list.append(perm)
        prev_ids = ids_t
        prev_perm = perm
    return perm_list

@torch.no_grad()
def visualize_tracked_masks(out, frames=8, slots=4):
    """
    out["masks"]: [B,T,K,1,H,W] (GPU)
    追跡して、行=固定slot、列=時間 で mask を表示
    """
    masks = out["masks"][0]  # [T,K,1,H,W]
    device = masks.device
    T, K = masks.shape[0], masks.shape[1]
    frames = min(frames, T)
    slots = min(slots, K)

    masks_T = masks[:frames]  # [F,K,1,H,W] GPU

    perm_list = track_slots_by_iou(masks_T)  # list length F

    fig, axes = plt.subplots(slots, frames, figsize=(2*frames, 2*slots))
    if slots == 1:
        axes = np.expand_dims(axes, 0)

    for t in range(frames):
        perm = perm_list[t]
        for s in range(slots):
            idx = perm[s]  # current frame slot index aligned to reference slot s
            m = masks_T[t, idx, 0]  # [H,W] GPU
            axes[s, t].imshow(m.detach().cpu().numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            axes[s, t].axis("off")
            if t == 0:
                axes[s, t].set_title(f"tracked slot {s}")
    plt.tight_layout()
    plt.show()

# --- 実行 ---
model.eval()
with torch.no_grad():
    out = model(x)  # x はGPU上にある想定

visualize_tracked_masks(out, frames=8, slots=4)

## 16. ベースライン評価プロトコル（FG-ARI / GT-mIoU / Recon / Tracking）

In [ ]:
import numpy as np
import torch
from scipy.optimize import linear_sum_assignment

@torch.no_grad()
def _hard_ids_from_soft_masks(m_soft_k1hw: torch.Tensor) -> torch.Tensor:
    """
    m_soft_k1hw: [K,1,H,W] or [K,H,W] soft masks (sum over K = 1)
    return: [H,W] hard assignment id map (0..K-1)
    """
    if m_soft_k1hw.ndim == 4:
        m_soft_khw = m_soft_k1hw[:, 0]  # [K,H,W]
    else:
        m_soft_khw = m_soft_k1hw
    return torch.argmax(m_soft_khw, dim=0)  # [H,W]

@torch.no_grad()
def _iou_matrix_from_hard_ids(ids_prev: torch.Tensor, ids_curr: torch.Tensor, K: int) -> torch.Tensor:
    """
    ids_prev, ids_curr: [H,W] with values in 0..K-1
    return: [K,K] IoU matrix (torch, on same device)
    """
    # one-hot: [H,W] -> [K,H,W]
    prev_oh = torch.nn.functional.one_hot(ids_prev.long(), num_classes=K).permute(2,0,1).bool()
    curr_oh = torch.nn.functional.one_hot(ids_curr.long(), num_classes=K).permute(2,0,1).bool()

    inter = (prev_oh[:, None] & curr_oh[None, :]).flatten(2).sum(-1).float()  # [K,K]
    union = (prev_oh[:, None] | curr_oh[None, :]).flatten(2).sum(-1).float()  # [K,K]
    return inter / (union + 1e-6)

@torch.no_grad()
def compute_id_swap_stats(masks_tk1hw: torch.Tensor, min_area: int = 25):
    """
    masks_tk1hw: [T,K,1,H,W] (SAVi out["masks"][b]) を想定
    return:
      dict: id_swap_mean, matched_iou_mean, valid_frac, per_t (optional)
    """
    assert masks_tk1hw.ndim == 5, f"expected [T,K,1,H,W], got {masks_tk1hw.shape}"
    T, K, _, H, W = masks_tk1hw.shape
    device = masks_tk1hw.device

    idx = np.arange(K, dtype=np.int64)  # prev slot index 0..K-1

    swap_list = []
    matched_iou_list = []
    valid_frac_list = []

    # t=0 の hard id
    ids_prev = _hard_ids_from_soft_masks(masks_tk1hw[0])  # [H,W] torch

    # prev slot area（hard割当の画素数）
    prev_area = torch.bincount(ids_prev.flatten(), minlength=K).float()  # [K]
    valid_prev = (prev_area >= float(min_area))  # torch bool [K]

    for t in range(1, T):
        ids_curr = _hard_ids_from_soft_masks(masks_tk1hw[t])  # [H,W]

        # IoU matrix
        iou = _iou_matrix_from_hard_ids(ids_prev, ids_curr, K)  # [K,K] torch
        iou_cpu = iou.detach().cpu().numpy()

        # Hungarian（最大IoU <=> 最小(1-IoU)）
        row_ind, col_ind = linear_sum_assignment(1.0 - iou_cpu)

        # ★ここが重要：perm を numpy 配列にする（listのままだとperm[mask]で落ちる）
        perm = np.asarray(col_ind, dtype=np.int64)  # perm[j] = matched curr slot for prev slot j

        valid_prev_np = valid_prev.detach().cpu().numpy().astype(bool)
        denom = max(int(valid_prev_np.sum()), 1)

        # ID swap：対応が identity からズレた割合
        swap = (perm[valid_prev_np] != idx[valid_prev_np]).sum() / denom

        # matched IoU：prev slot j と perm[j] の IoU を取って平均（validのみ）
        matched = iou_cpu[idx, perm]  # shape [K]（要素ごとの対応IoU）
        matched_iou = float(matched[valid_prev_np].mean()) if valid_prev_np.any() else float(matched.mean())

        swap_list.append(float(swap))
        matched_iou_list.append(float(matched_iou))
        valid_frac_list.append(float(valid_prev_np.mean()))

        # 次へ更新
        ids_prev = ids_curr
        prev_area = torch.bincount(ids_prev.flatten(), minlength=K).float()
        valid_prev = (prev_area >= float(min_area))

    return {
        "id_swap_mean": float(np.mean(swap_list)) if len(swap_list) else 0.0,
        "matched_iou_mean": float(np.mean(matched_iou_list)) if len(matched_iou_list) else 0.0,
        "valid_frac_mean": float(np.mean(valid_frac_list)) if len(valid_frac_list) else 0.0,
        "id_swap_per_t": swap_list,
        "matched_iou_per_t": matched_iou_list,
    }

## 17. 長期記憶評価プロトコル（ReID after occlusion / Long-horizon rollout / Ablation gap）

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from scipy.optimize import linear_sum_assignment


def _get_batch(batch):
    # loader が tuple/list を返す前提。返り値がx単体でも対応。
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        gt = batch[1] if len(batch) > 1 else None
    else:
        x, gt = batch, None
    return x, gt


def _init_slots(model, B, device):
    # SAVi実装差の吸収：init_slots の場所が違う場合に対応
    if hasattr(model, "corrector") and hasattr(model.corrector, "init_slots"):
        return model.corrector.init_slots(B, device=device)
    if hasattr(model, "init_slots"):
        return model.init_slots(B, device=device)
    raise AttributeError("init_slots が見つかりません：model.corrector.init_slots または model.init_slots を確認してください。")


def _encode(model, x_t):
    out = model.encoder(x_t)
    # encoder が (feats, pos) のように返す場合
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def _correct(model, feats, slots):
    # corrector の引数名が slots_init の場合とそうでない場合を吸収
    try:
        slots_new, attn = model.corrector(feats, slots_init=slots)
    except TypeError:
        slots_new, attn = model.corrector(feats, slots)
    return slots_new, attn


def _decode(model, slots):
    out = model.decoder(slots)
    # decoder が tuple(recon, masks, rgb) を返す前提（違う場合はここを調整）
    if isinstance(out, (tuple, list)) and len(out) >= 2:
        recon, masks = out[0], out[1]
        return recon, masks
    if isinstance(out, dict) and "recon" in out and "masks" in out:
        return out["recon"], out["masks"]
    raise TypeError("decoder の返り値形式が想定と異なります。model.decoder の返り値を確認してください。")


@torch.no_grad()
def _run_no_memory_inference(model, x):
    """
    記憶なしモード：毎フレーム slots を初期化して correct のみで推論（transitionを使わない）
    x: [B,T,3,H,W]
    return: recon [B,T,3,H,W], masks [B,T,K,1,H,W]
    """
    B,T,_,_,_ = x.shape
    recons, masks_all = [], []
    for t in range(T):
        feats = _encode(model, x[:,t])
        slots = _init_slots(model, B, x.device)          # 毎フレーム初期化
        slots, _ = _correct(model, feats, slots)
        recon_t, masks_t = _decode(model, slots)         # recon_t: [B,3,H,W], masks_t: [B,K,1,H,W]
        recons.append(recon_t)
        masks_all.append(masks_t)
    recon = torch.stack(recons, dim=1)
    masks = torch.stack(masks_all, dim=1)
    return recon, masks


@torch.no_grad()
def _run_partial_observation_rollout(model, x, t_obs=4, no_memory=False):
    """
    観測停止後ロールアウト：
      t < t_obs: correctorで観測を取り込む
      t >= t_obs:
        - no_memory=False: transitionのみで予測（=長期記憶＋ダイナミクス）
        - no_memory=True : 毎ステップ slots を初期化（=記憶なし比較。ほぼ無情報予測）
    return: recon [B,T,3,H,W], masks [B,T,K,1,H,W]
    """
    B,T,_,_,_ = x.shape
    slots = _init_slots(model, B, x.device)

    recons, masks_all = [], []
    for t in range(T):
        if t > 0:
            if no_memory and (t >= t_obs):
                slots = _init_slots(model, B, x.device)  # 観測停止後は毎回初期化（記憶なし）
            else:
                slots = model.transition(slots)

        if t < t_obs:
            feats = _encode(model, x[:,t])
            slots, _ = _correct(model, feats, slots)

        recon_t, masks_t = _decode(model, slots)
        recons.append(recon_t)
        masks_all.append(masks_t)

    recon = torch.stack(recons, dim=1)
    masks = torch.stack(masks_all, dim=1)
    return recon, masks


@torch.no_grad()
def _masks_to_slot_labels(masks_bt_k1hw):
    # masks: [B,T,K,1,H,W] -> labels [B,T,H,W]
    return masks_bt_k1hw.squeeze(3).argmax(dim=2)


@torch.no_grad()
def _hungarian_map_gt_to_slot(pred_lbl_hw, gt_id_hw, bg_id=0, min_area=10, max_gt=30):
    """
    1フレームで GT instance -> slot の対応を作る（IoU最大化 Hungarian）
    戻り値: dict(gt_id -> slot_id), present_gt_set
    """
    # present GT (area>=min_area)
    ids, counts = torch.unique(gt_id_hw, return_counts=True)
    present = []
    for gid, cnt in zip(ids.tolist(), counts.tolist()):
        if gid == bg_id:
            continue
        if cnt >= min_area:
            present.append(int(gid))
    present_set = set(present)
    if len(present) == 0:
        return {}, set()

    # build IoU matrix [K,M]
    gt_ids = present[:max_gt]
    K = int(pred_lbl_hw.max().item()) + 1
    M = len(gt_ids)

    pred_oh = F.one_hot(pred_lbl_hw, num_classes=K).permute(2,0,1).float()  # [K,H,W]
    gt_oh = torch.stack([(gt_id_hw == gid).float() for gid in gt_ids], dim=0)  # [M,H,W]

    pred_flat = pred_oh.reshape(K, -1)
    gt_flat = gt_oh.reshape(M, -1)

    inter = pred_flat @ gt_flat.t()
    area_p = pred_flat.sum(dim=1, keepdim=True)
    area_g = gt_flat.sum(dim=1, keepdim=True).t()
    union = area_p + area_g - inter
    iou = inter / (union + 1e-6)

    iou_np = iou.detach().cpu().numpy()
    row, col = linear_sum_assignment(-iou_np)

    mapping = {}
    for r, c in zip(row.tolist(), col.tolist()):
        mapping[int(gt_ids[c])] = int(r)
    return mapping, present_set


@torch.no_grad()
def _build_framewise_mapping(pred_lbl_bt_hw, gt_bt_hw, bg_id=0, min_area=10):
    """
    mapping[b][t][gt_id] = slot_id
    present[b][t] = set(gt_id)
    """
    B,T,_,_ = pred_lbl_bt_hw.shape
    mapping = [[{} for _ in range(T)] for _ in range(B)]
    present = [[set() for _ in range(T)] for _ in range(B)]
    for b in range(B):
        for t in range(T):
            m, p = _hungarian_map_gt_to_slot(pred_lbl_bt_hw[b,t], gt_bt_hw[b,t],
                                             bg_id=bg_id, min_area=min_area)
            mapping[b][t] = m
            present[b][t] = p
    return mapping, present


@torch.no_grad()
def _reid_after_gap(mapping, present, gap_min=5):
    """
    gap>=gap_min の不在を挟んで再登場したとき、同じslotに戻れた割合
    """
    B = len(mapping)
    T = len(mapping[0])

    total = 0
    correct = 0
    gaps = []

    for b in range(B):
        all_ids = set()
        for t in range(T):
            all_ids |= present[b][t]
        for gid in all_ids:
            vis = [gid in present[b][t] for t in range(T)]
            t = 0
            while t < T:
                if not vis[t]:
                    t += 1
                    continue
                # visible run
                t0 = t
                while t < T and vis[t]:
                    t += 1
                t1 = t  # first invisible
                # invisible gap
                t2 = t1
                while t2 < T and not vis[t2]:
                    t2 += 1
                gap = t2 - t1

                # event: visible -> long gap -> visible
                if t2 < T and gap >= gap_min:
                    pre_slot = mapping[b][t1-1].get(gid, None)
                    post_slot = mapping[b][t2].get(gid, None)
                    if (pre_slot is not None) and (post_slot is not None):
                        total += 1
                        gaps.append(gap)
                        if pre_slot == post_slot:
                            correct += 1
                t = t2

    rate = correct / max(1, total)
    avg_gap = float(np.mean(gaps)) if len(gaps) else float("nan")
    return {"events": int(total), "rate": float(rate), "avg_gap": avg_gap}


@torch.no_grad()
def _mse_psnr(recon_bt_3hw, x_bt_3hw, t_from):
    mse = F.mse_loss(recon_bt_3hw[:, t_from:], x_bt_3hw[:, t_from:]).item()
    psnr = 20.0*np.log10(2.0) - 10.0*np.log10(max(mse, 1e-12))
    return float(mse), float(psnr)


@torch.no_grad()
def run_longterm_memory_suite(
    model,
    loader,
    n_batches=20,
    gap_list=(5,10,20),
    t_obs_list=(2,4,8),
    bg_id=0,
    min_area=10,
):
    """
    戻り値：
      dict with keys:
        - reid_table: DataFrame（gapごとの mem/no-mem/Δ）
        - rollout_table: DataFrame（t_obsごとの mem/no-mem/Δ）
    """
    model.eval()

    # accumulators: gap -> list of rates
    reid_mem = {g: [] for g in gap_list}
    reid_nom = {g: [] for g in gap_list}
    reid_events_mem = {g: 0 for g in gap_list}
    reid_events_nom = {g: 0 for g in gap_list}

    # rollout: t_obs -> list of mse/psnr
    roll_mem = {t: {"mse": [], "psnr": []} for t in t_obs_list}
    roll_nom = {t: {"mse": [], "psnr": []} for t in t_obs_list}

    for i, batch in enumerate(loader):
        if i >= n_batches:
            break

        x, gt = _get_batch(batch)
        x = x.to(device)
        if gt is not None:
            gt = gt.to(device)

        # ---------- ReID (requires GT) ----------
        if gt is not None:
            # memory mode: use model(x) masks
            out = model(x)
            pred_lbl_mem = _masks_to_slot_labels(out["masks"])  # [B,T,H,W]
            map_m, pres_m = _build_framewise_mapping(pred_lbl_mem, gt, bg_id=bg_id, min_area=min_area)

            # no-memory mode: each frame reset
            recon_nomem, masks_nomem = _run_no_memory_inference(model, x)
            pred_lbl_nom = _masks_to_slot_labels(masks_nomem)
            map_n, pres_n = _build_framewise_mapping(pred_lbl_nom, gt, bg_id=bg_id, min_area=min_area)

            for g in gap_list:
                rm = _reid_after_gap(map_m, pres_m, gap_min=g)
                rn = _reid_after_gap(map_n, pres_n, gap_min=g)

                reid_events_mem[g] += rm["events"]
                reid_events_nom[g] += rn["events"]
                if rm["events"] > 0:
                    reid_mem[g].append(rm["rate"])
                if rn["events"] > 0:
                    reid_nom[g].append(rn["rate"])

        # ---------- Rollout ----------
        for t_obs in t_obs_list:
            recon_roll, _ = _run_partial_observation_rollout(model, x, t_obs=t_obs, no_memory=False)
            mse_m, psnr_m = _mse_psnr(recon_roll, x, t_from=t_obs)
            roll_mem[t_obs]["mse"].append(mse_m)
            roll_mem[t_obs]["psnr"].append(psnr_m)

            recon_roll_nm, _ = _run_partial_observation_rollout(model, x, t_obs=t_obs, no_memory=True)
            mse_n, psnr_n = _mse_psnr(recon_roll_nm, x, t_from=t_obs)
            roll_nom[t_obs]["mse"].append(mse_n)
            roll_nom[t_obs]["psnr"].append(psnr_n)

    # ---------- Build tables ----------
    def _mean(xs):
        return float(np.mean(xs)) if len(xs) else float("nan")

    # ReID table
    reid_rows = []
    for g in gap_list:
        mem = _mean(reid_mem[g])
        nom = _mean(reid_nom[g])
        reid_rows.append({
            "gap_min": g,
            "ReID_rate_mem": mem,
            "ReID_rate_no_mem": nom,
            "ΔReID(mem-no_mem)": (mem - nom) if (not np.isnan(mem) and not np.isnan(nom)) else float("nan"),
            "events_mem": int(reid_events_mem[g]),
            "events_no_mem": int(reid_events_nom[g]),
        })
    reid_table = pd.DataFrame(reid_rows)

    # Rollout table
    roll_rows = []
    for t_obs in t_obs_list:
        mem_mse = _mean(roll_mem[t_obs]["mse"])
        nom_mse = _mean(roll_nom[t_obs]["mse"])
        mem_psnr = _mean(roll_mem[t_obs]["psnr"])
        nom_psnr = _mean(roll_nom[t_obs]["psnr"])
        roll_rows.append({
            "t_obs": t_obs,
            "Rollout_MSE_mem": mem_mse,
            "Rollout_MSE_no_mem": nom_mse,
            "ΔRollout_MSE(no_mem - mem)": (nom_mse - mem_mse) if (not np.isnan(mem_mse) and not np.isnan(nom_mse)) else float("nan"),
            "Rollout_PSNR_mem": mem_psnr,
            "Rollout_PSNR_no_mem": nom_psnr,
        })
    rollout_table = pd.DataFrame(roll_rows)

    return {"reid_table": reid_table, "rollout_table": rollout_table}


# ---- 実行例 ----
if val_loader is None:
    print("[WARN] val_loader が見つかりません。val_loader を用意してから実行してください。")
else:
    results = run_longterm_memory_suite(
        model, val_loader,
        n_batches=20,                # 重ければ 5〜10
        gap_list=[5,10,20],
        t_obs_list=[2,4,8],
        bg_id=0,
        min_area=10,
    )
    print("\n[ReID after occlusion]")
    display(results["reid_table"])
    print("\n[Long-horizon rollout]")
    display(results["rollout_table"])